# Importando as Bibliotecas

In [ ]:
import pandas as pd
import duckdb as db
from datetime import datetime
from IPython.display import HTML, display
from datetime import datetime, date
from openpyxl import Workbook, load_workbook

# Carregamento da base de controle de processos

In [ ]:
id = 19

path_registros_processos = r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'
registros_processos = pd.read_excel(path_registros_processos, sheet_name = "REGISTROS", engine='openpyxl')
wb_p = load_workbook(path_registros_processos)
ws_p = wb_p['REGISTROS']

# Controle de atualização de processo: Etapa 0
tempo_0 = [id, datetime.today(), 0]
ws_p.append(tempo_0)
wb_p.save(path_registros_processos)

# Inicializando as variáveis de Controle

In [ ]:
hoje = datetime.today().date()
data_br = hoje.strftime('%d/%m/%Y') # Formato DD/MM/AAAA

# Padrão de cores das tabelas (mantido do seu código)
css = """
<style>
    .minha-tabela {
        border-collapse: collapse;
        width: 50%;
        margin: auto;
    }
    .minha-tabela th, .minha-tabela td {
        border: 1px solid #ddd;
        padding: 8px;
    }
    .minha-tabela th {
        background-color: #377AB4;
        color: white;
    }
    .minha-tabela tr:nth-child(even) {
        background-color: #f2f2f2;
    }
    .minha-tabela tr:hover {
        background-color: #ddd;
    }
</style>
"""

# Variável com o diretório do arquivo HTML a ser salvo
caminho_arquivo = r'X:\Gestao_de_Pessoas\Analytics\11 - Validações\11.1 - Cargos e Salários\CARGOS E SALÁRIOS - ' + data_br.replace('/', '-') + '.html'

# Importação das Bases

In [ ]:
caminho_cargos_datamace = r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\CARGOS_DATAMACE.xlsx'
caminho_cargos_salarios = r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\CARGOS E SALÁRIOS.xlsx'
caminho_colaboradores = r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\COLABORADORES.xlsx'

# Carregar bases
cargos_datamace = pd.read_excel(caminho_cargos_datamace)
cargos_salarios = pd.read_excel(caminho_cargos_salarios, usecols='B:L', skiprows=3)
colaboradores = pd.read_excel(caminho_colaboradores)

# Início da Limpeza e Verificação Detalhada

In [ ]:
# Limpeza e Verificação da base cargos_datamace
print("--- Verificação antes da limpeza de cargos_datamace ---")
print(f"Tipo original da coluna 'id_Cargo': {cargos_datamace['id_Cargo'].dtype}")
print("Exemplos de valores em 'id_Cargo' antes da limpeza:")
print(cargos_datamace['id_Cargo'].value_counts(dropna=False).head(10)) # Mostra os 10 valores mais frequentes, incluindo NaN

# Aplica pd.to_numeric com errors='coerce' para converter tudo que pode para número.
# Valores que não podem ser convertidos (como '1218G') se tornam NaN (Not a Number).
cargos_datamace['id_Cargo'] = pd.to_numeric(cargos_datamace['id_Cargo'], errors='coerce')

# Remove as linhas onde 'id_Cargo' se tornou NaN (ou seja, eram valores não numéricos)
cargos_datamace = cargos_datamace.dropna(subset=['id_Cargo'])

# Converte a coluna para o tipo inteiro (int)
cargos_datamace['id_Cargo'] = cargos_datamace['id_Cargo'].astype(int)

print("\n--- Verificação após a limpeza de cargos_datamace ---")
print(f"Tipo final da coluna 'id_Cargo': {cargos_datamace['id_Cargo'].dtype}")
print("Exemplos de valores em 'id_Cargo' após a limpeza:")
print(cargos_datamace['id_Cargo'].value_counts(dropna=False).head(10)) # Deve conter apenas números
if cargos_datamace['id_Cargo'].apply(lambda x: not isinstance(x, (int, float))).any():
    print("AVISO: Ainda existem valores não numéricos em 'id_Cargo' após a limpeza!")
else:
    print("Sucesso: 'id_Cargo' parece estar limpo e numérico.")


# Limpeza e Verificação cargos_salarios (Coluna "ID Datamace")
print("\n--- Verificação antes da limpeza de cargos_salarios['ID Datamace'] ---")
print(f"Tipo original da coluna 'ID Datamace': {cargos_salarios['ID Datamace'].dtype}")
print("Exemplos de valores em 'ID Datamace' antes da limpeza:")
print(cargos_salarios['ID Datamace'].value_counts(dropna=False).head(10))

cargos_salarios['ID Datamace'] = pd.to_numeric(cargos_salarios['ID Datamace'], errors='coerce')
cargos_salarios = cargos_salarios.dropna(subset=['ID Datamace'])
cargos_salarios['ID Datamace'] = cargos_salarios['ID Datamace'].astype(int)

print("\n--- Verificação após a limpeza de cargos_salarios['ID Datamace'] ---")
print(f"Tipo final da coluna 'ID Datamace': {cargos_salarios['ID Datamace'].dtype}")
print("Exemplos de valores em 'ID Datamace' após a limpeza:")
print(cargos_salarios['ID Datamace'].value_counts(dropna=False).head(10))
if cargos_salarios['ID Datamace'].apply(lambda x: not isinstance(x, (int, float))).any():
    print("AVISO: Ainda existem valores não numéricos em 'ID Datamace' após a limpeza!")
else:
    print("Sucesso: 'ID Datamace' parece estar limpo e numérico.")

print("\n--- Fim das verificações do Pandas ---")

# Criar ou substituir as tabelas no DuckDB com os DataFrames limpos
db.sql("CREATE OR REPLACE TABLE cargos_datamace AS SELECT * FROM cargos_datamace")
db.sql("CREATE OR REPLACE TABLE cargos_salarios AS SELECT * FROM cargos_salarios")
db.sql("CREATE OR REPLACE TABLE colaboradores AS SELECT * FROM colaboradores") # Também para colaboradores, para garantir consistência

# VERIFICAÇÃO
print("\n--- Verificação de tipos após CREATE OR REPLACE ---")
print("Esquema de cargos_datamace:")
print(db.query("PRAGMA table_info('cargos_datamace');").df())
print("\nEsquema de cargos_salarios:")
print(db.query("PRAGMA table_info('cargos_salarios');").df())
print("--- Fim das verificações ---")

# Validações de Dados

In [ ]:
# 1. Cargos cadastrados após o reenquadramento e que não estão no controle de Gestão de Pessoas
base_nao_cad_controle = db.query("""
    SELECT COUNT(*)
    FROM cargos_datamace AS A
    LEFT JOIN cargos_salarios AS B
        ON A.id_Cargo = B."ID Datamace"
    WHERE B."ID Datamace" IS NULL
      AND A.id_Cargo >= 834
""")
nao_cad_controle = int(base_nao_cad_controle.df().iloc[0, 0])  # Resultado

# Tabela em HTML para cargos não cadastrados no controle
html_nao_cad_controle_tabela = db.query("""
    SELECT
        A.id_Cargo AS "ID Datamace",
        A.cargo_completo AS "Cargo Datamace"
    FROM cargos_datamace AS A
    LEFT JOIN cargos_salarios AS B
        ON A.id_Cargo = B."ID Datamace"
    WHERE B."ID Datamace" IS NULL
      AND A.id_Cargo >= 834
""").df().to_html(index=False, classes='minha-tabela')

# 2. Cargos anteriores ao reenquadramento que ainda estão ativos
base_antigos_ativos = db.query("""
    SELECT COUNT(*)
    FROM cargos_datamace
    WHERE id_Cargo < 834
      AND Status = 'Ativo'
""")
antigos_ativos = int(base_antigos_ativos.df().iloc[0, 0])  # Resultado

# Tabela em HTML para cargos antigos ativos
html_cargos_antigos_ativos_tabela = db.query("""
    SELECT
        id_Cargo         AS "ID Datamace",
        cargo_completo   AS Cargo
    FROM
        cargos_datamace
    WHERE
        id_Cargo < 834
    AND
        Status = 'Ativo'
""").df().to_html(index=False, classes='minha-tabela') # Adicionei a classe CSS

# 3. Cargos que tiveram alteração na descrição completa do cargo
base_descricao_completa = db.query("""
    SELECT COUNT(*)
    FROM cargos_datamace AS A
    LEFT JOIN cargos_salarios AS B
        ON A.id_Cargo = B."ID Datamace"
    WHERE B."ID Datamace" IS NOT NULL -- Apenas se o ID corresponde
      AND A.id_Cargo >= 834
      AND A.Cargo_completo <> B."Descrição Completa"
""")
descricao_completa = int(base_descricao_completa.df().iloc[0, 0])

# Tabela em HTML para cargos com descrição completa alterada
html_descricao_completa_tabela = db.query("""
    SELECT
        A.id_Cargo AS "ID Datamace",
        A.Cargo_completo AS "Descrição Completa Datamace",
        B."Descrição Completa" AS "Descrição Completa RH"
    FROM cargos_datamace AS A
    LEFT JOIN cargos_salarios AS B
        ON A.id_Cargo = B."ID Datamace"
    WHERE B."ID Datamace" IS NOT NULL
      AND A.id_Cargo >= 834
      AND A.Cargo_completo <> B."Descrição Completa"
""").df().to_html(index=False, classes='minha-tabela')

# 4. Cargos que tiveram alteração na descrição resumida do cargo
base_descricao_resumida = db.query("""
    SELECT COUNT(*)
    FROM cargos_datamace AS A
    LEFT JOIN cargos_salarios AS B
        ON A.id_Cargo = B."ID Datamace"
    WHERE B."ID Datamace" IS NOT NULL -- Apenas se o ID corresponde
      AND A.id_Cargo >= 834
      AND A.cargo_abreviado <> B."Descrição Resumida"
""")
descricao_resumida = int(base_descricao_resumida.df().iloc[0, 0])

# Tabela em HTML para cargos com descrição resumida alterada
html_descricao_resumida_tabela = db.query("""
    SELECT
        A.id_Cargo AS "ID Datamace",
        A.cargo_abreviado AS "Descrição Resumida Datamace",
        B."Descrição Resumida" AS "Descrição Resumida RH"
    FROM cargos_datamace AS A
    LEFT JOIN cargos_salarios AS B
        ON A.id_Cargo = B."ID Datamace"
    WHERE B."ID Datamace" IS NOT NULL
      AND A.id_Cargo >= 834
      AND A.cargo_abreviado <> B."Descrição Resumida"
""").df().to_html(index=False, classes='minha-tabela')

# 5. Colaboradores com o salário divergente do salário cadastrado na tabela de cargos e salários
base_salario_dif = db.query("""
    SELECT
            A.registro         AS "Matrícula",
            A.nome             AS "Nome do colaborador",
            A.cargo_completo   AS "Cargo",
            A.centro_custo     AS "Centro de custo",
            A.salario_atual    AS "Salário praticado",
            B."Salário"        AS "Salário da tabela",
            A.hora_base        AS "Carga horária"
    FROM
            colaboradores AS A
    LEFT JOIN
            cargos_salarios AS B
        ON
            A.cargo_completo = B."Descrição Completa"
    WHERE
            A.salario_atual <> B."Salário"
        AND
            A.descricao_rescisao = 'ATIVO'
    ORDER BY
            A.cargo_completo
""").df().to_html(index=False, classes='minha-tabela') # Adicionei a classe CSS

salario_dif_count = db.query("""
    SELECT COUNT() AS Quantidade
    FROM colaboradores AS A
    LEFT JOIN cargos_salarios AS B
        ON A.cargo_completo = B."Descrição Completa"
    WHERE A.salario_atual <> B."Salário"
      AND A.descricao_rescisao = 'ATIVO'
""").df()

salario_dif = int(salario_dif_count.iloc[0, 0])

# Geração do HTML

In [ ]:
# Cabeçalho
html_titulo = f"""
<div style="background-color: #23A638; padding: 1px; border: 3px solid #23A638;">
    <h1 style="color: #FFFFFF; font-size: 28px; font-family: 'Verdana'; font-weight: bold;"> VALIDAÇÃO DE CARGOS E SALÁRIOS - {data_br}</h1>
</div>
"""

# Validação 1
html_val_1 = f"""
<div style="background-color: #FFFFFF; padding: 10px; border: 3px solid #23A638; height: 30px;">
    <p style="font-size: 18px; font-family: 'Verdana'; font-weight: bold;">Quantidade de cargos cadastrados após reenquadramento que não estão no controle GP: {nao_cad_controle}</p>
</div>
<p>{html_nao_cad_controle_tabela}</p>
"""

# Validação 2
html_val_2 = f"""
<div style="background-color: #FFFFFF; padding: 10px; border: 3px solid #23A638; height: 30px;">
    <p style="font-size: 18px; font-family: 'Verdana'; font-weight: bold;">Quantidade de cargos antigos (anteriores ao reenquadramento) que ainda estão ativos: {antigos_ativos}</p>
</div>
<p>{html_cargos_antigos_ativos_tabela}</p>
"""

# Validação 3
html_val_3 = f"""
<div style="background-color: #FFFFFF; padding: 10px; border: 3px solid #23A638; height: 30px;">
    <p style="font-size: 18px; font-family: 'Verdana'; font-weight: bold;">Quantidade de cargos que tiveram a descrição completa alterada: {descricao_completa}</p>
</div>
<p>{html_descricao_completa_tabela}</p>
"""

# Validação 4
html_val_4 = f"""
<div style="background-color: #FFFFFF; padding: 10px; border: 3px solid #23A638; height: 30px;">
    <p style="font-size: 18px; font-family: 'Verdana'; font-weight: bold;">Quantidade de cargos que tiveram a descrição resumida alterada: {descricao_resumida}</p>
</div>
<p>{html_descricao_resumida_tabela}</p>
"""

# Validação 5
html_val_5 = f"""
<div style="background-color: #FFFFFF; padding: 10px; border: 3px solid #23A638; height: 30px;">
    <p style="font-size: 18px; font-family: 'Verdana'; font-weight: bold;">Quantidade de colaboradores com o salário divergente da tabela: {salario_dif}</p>
</div>
<p>{base_salario_dif}</p>
"""

# Código HTML final
html_resumo = css + '<br>' + html_titulo + '<br>' + html_val_1 + '<br>' + html_val_2 + '<br>' + html_val_3 + '<br>' + html_val_4 + '<br>' + html_val_5 + '<br>'

# Exibindo código HTML
display(HTML(html_resumo))

# Gerando arquivo HTML e salvando arquivo
with open(caminho_arquivo, 'w', encoding='utf-8') as file:
    file.write(html_resumo)

In [ ]:
tempo_1 = [id, datetime.today(), 1]

print('----------------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('     ⏱️   Tempo de execução:')
print('')
print(f'   {tempo_1[1] - tempo_0[0]}')
print('')
print('----------------------------------------------------------------------------------------------------')